# m06_l02_replay

Exobean Engineering. Run the cells from top to bottom.

In [ ]:
# Finds the code companion. On Google Colab it downloads it once.
import glob, io, os, sys, urllib.request, zipfile
URL = "https://github.com/exobean/ai-agents-level1/releases/latest/download/shopassist-companion.zip"
if not os.path.isdir("shopassist"):
    if os.path.isdir(os.path.join("..", "shopassist")):
        os.chdir("..")  # opened from the notebooks folder on your computer
    else:
        found = glob.glob("*-companion")  # kept: your edited answers stay
        if not found and "google.colab" in sys.modules:
            try:
                data = io.BytesIO(urllib.request.urlopen(URL).read())
            except OSError:  # network blocks the download: upload the zip
                from google.colab import files
                data = next(iter(files.upload()))
            zipfile.ZipFile(data).extractall(".")
            found = glob.glob("*-companion")
        if found:
            os.chdir(found[0])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Companion ready" if os.path.isdir("shopassist") else
      "Companion not found: see Lesson 0.1 of the book")


## Replay

In [ ]:
import tempfile
from pathlib import Path

from shopassist.app_v2 import new_session
from shopassist.cassette import RecordingModel, ReplayMiss, ReplayModel
from shopassist.models import TeachingModel

tape = Path(tempfile.mkdtemp()) / "cassette.json"

# 1. Record once (in real life: the real, paid model).
rec, _ = new_session(RecordingModel(TeachingModel(), tape))
first = rec.ask("Where is my order #1042?")

# 2. Replay in every test run: no network, no cost, same answer.
replay, _ = new_session(ReplayModel(tape))
again = replay.ask("Where is my order #1042?")
print("replay matches:", again == first)

# 3. A changed conversation is caught, not silently answered.
try:
    new_session(ReplayModel(tape))[0].ask("Where is order #1043?")
except ReplayMiss as miss:
    print("replay miss:   ", miss)
